# Titled Tuesday Command Center
Run top-to-bottom every Tuesday morning. Update the config cell first.

In [ ]:
# ── Weekly Config — update these each Tuesday ─────────────────────────────────
TOURN_DATE = "2026-08-05"   # Next Tuesday's date (YYYY-MM-DD)
BANKROLL   = 2000           # Total bankroll in USD
KELLY_MULT = 0.5            # Kelly fraction (0.5 = half-Kelly recommended)

EVENT_TICKERS = [
    "KXTITLEDTUESDAY-26AUG05",
    "KXTITLEDTUESTOP-26AUG05T3",
    "KXTITLEDTUESTOP-26AUG05T5",
    "KXTITLEDTUESTOP-26AUG05T8",
]

In [ ]:
import sys
sys.path.insert(0, '..')

import re
import sqlite3
import time
import warnings

import numpy as np
import pandas as pd
import requests

from src.config import DB_PATH
from src.data import get_username_mappings
from src.pipeline import run_predictions, save_predictions
from src.portfolio import (
    build_portfolio, run_portfolio_mc_score, pnl_summary,
    run_payoff_matrix, portfolio_kelly,
)

warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 120)

USERNAME_TO_PLAYER, PLAYER_TO_USERNAME = get_username_mappings()
print('Imports OK')

---
## 1 — Data freshness

In [ ]:
conn = sqlite3.connect(DB_PATH)
last_tourn, last_date_raw = conn.execute(
    'SELECT title, MAX(date) FROM titled_tuesday_tournaments'
).fetchone()
recent = pd.read_sql_query(
    'SELECT date, title, num_players, winner FROM titled_tuesday_tournaments '
    'ORDER BY date DESC LIMIT 3', conn
)
conn.close()

days_ago = (pd.Timestamp.now() - pd.Timestamp(last_date_raw)).days
status = 'OK' if days_ago <= 8 else 'STALE — run scripts/update_tt_json.py'
print(f'Latest data: {last_date_raw}  ({days_ago}d ago)  [{status}]\n')
display(recent)

---
## 2 — Run predictions

In [ ]:
run = run_predictions(TOURN_DATE)
results = run.results
print(f'\nPool: {len(results):,} players')

In [ ]:
# Uncomment to persist predictions to DB (do this after finalising adjustments)
# save_predictions(run, TOURN_DATE)

---
## 3 — Attendance adjustment editor

View active adjustments, activate auto-suggestions, and add your own overrides or nudges.

**After any change, re-run Section 2 to apply the new adjustments.**

In [ ]:
# ── Display current adjustments for TOURN_DATE ────────────────────────────────
conn = sqlite3.connect(DB_PATH)
df_adj = pd.read_sql_query(
    "SELECT id, username, type, value, source, reason, is_active, created_at "
    "FROM attendance_adjustments WHERE tourn_date = ? "
    "ORDER BY is_active DESC, type, username",
    conn, params=(TOURN_DATE,)
)
conn.close()

if df_adj.empty:
    print(f'No adjustments found for {TOURN_DATE}.')
    print('Run:  python scripts/migrate_adjustments.py  to seed from the default lists.')
else:
    n_active = df_adj['is_active'].sum()
    n_suggested = (~df_adj['is_active'].astype(bool)).sum()
    print(f'{n_active} active  |  {n_suggested} suggested (is_active=0)\n')

    def _highlight(row):
        if not row['is_active']:
            return ['color: #aaa'] * len(row)
        if row['type'] == 'nudge':
            return ['background-color: #e8f4fd'] * len(row)
        if row['value'] == 0.0:
            return ['background-color: #fdecea'] * len(row)
        return ['background-color: #fff3cd'] * len(row)

    display(
        df_adj.style
        .apply(_highlight, axis=1)
        .format({'value': '{:.3f}'})
        .set_caption(f'Adjustments for {TOURN_DATE} | blue=nudge  red=cut  yellow=override  grey=suggested')
    )

In [ ]:
# ── Helper functions — call these cells to manage adjustments ─────────────────

def add_override(username, p, reason='', source='manual'):
    """Set a player's final p_participate to p (0.0–1.0), bypassing the model."""
    _upsert('override', username, float(p), reason, source)

def add_nudge(username, delta, reason='', source='manual'):
    """Shift a player's log-odds by delta.  +0.5 ≈ +10–15pp at moderate base rates."""
    _upsert('nudge', username, float(delta), reason, source)

def cut(username, reason='', source='manual'):
    """Set p_participate = 0 for this player."""
    add_override(username, 0.0, reason=reason, source=source)

def activate(row_id):
    """Activate a single suggested row (is_active=0 → 1)."""
    _set_active(row_id, 1)
    print(f'Activated row id={row_id}')

def deactivate(row_id):
    """Deactivate an adjustment without deleting it (keeps history)."""
    _set_active(row_id, 0)
    print(f'Deactivated row id={row_id}')

def activate_all_suggestions():
    """Activate all is_active=0 rows for TOURN_DATE."""
    conn = sqlite3.connect(DB_PATH)
    n = conn.execute(
        'UPDATE attendance_adjustments SET is_active=1 WHERE tourn_date=? AND is_active=0',
        (TOURN_DATE,)
    ).rowcount
    conn.commit(); conn.close()
    print(f'Activated {n} suggestion(s)')

def _upsert(adj_type, username, value, reason, source):
    conn = sqlite3.connect(DB_PATH)
    existing = conn.execute(
        'SELECT id FROM attendance_adjustments WHERE username=? AND tourn_date=? AND type=?',
        (username, TOURN_DATE, adj_type)
    ).fetchone()
    if existing:
        conn.execute(
            'UPDATE attendance_adjustments SET value=?, reason=?, source=?, is_active=1 WHERE id=?',
            (value, reason, source, existing[0])
        )
        print(f'Updated {adj_type} for {username}: value={value}')
    else:
        conn.execute(
            'INSERT INTO attendance_adjustments '
            '(username, tourn_date, type, value, source, reason, is_active) '
            'VALUES (?, ?, ?, ?, ?, ?, 1)',
            (username, TOURN_DATE, adj_type, value, source, reason)
        )
        print(f'Added {adj_type} for {username}: value={value}')
    conn.commit(); conn.close()

def _set_active(row_id, flag):
    conn = sqlite3.connect(DB_PATH)
    conn.execute('UPDATE attendance_adjustments SET is_active=? WHERE id=?', (flag, row_id))
    conn.commit(); conn.close()

print('Helper functions loaded.')
print('  add_override(username, p, reason)   — set final probability')
print('  add_nudge(username, delta, reason)  — log-odds shift (+0.5 ≈ +10–15pp)')
print('  cut(username, reason)               — set p=0')
print('  activate(row_id) / deactivate(row_id)')
print('  activate_all_suggestions()')

In [ ]:
# ── Example: add a manual override for Magnus ─────────────────────────────────
# Uncomment and edit, then re-run Section 2 above.

# add_override(
#     PLAYER_TO_USERNAME.get('Magnus Carlsen', 'MagnusCarlsen'),
#     p=0.45,
#     reason='Post-Superbet rest but historically opportunistic'
# )

# add_nudge(
#     PLAYER_TO_USERNAME.get('Denis Lazavik', 'DenisLazavik'),
#     delta=0.3,
#     reason='Hot streak, likely motivated'
# )

# cut(
#     PLAYER_TO_USERNAME.get('Hikaru Nakamura', 'Hikaru'),
#     reason='Saint Louis Rapid & Blitz'
# )

---
## 4 — Top predictions table

Yellow rows had a manual adjustment applied.

In [ ]:
# Fetch adjusted usernames for highlighting
conn = sqlite3.connect(DB_PATH)
adj_rows = conn.execute(
    'SELECT username FROM attendance_adjustments WHERE tourn_date=? AND is_active=1',
    (TOURN_DATE,)
).fetchall()
conn.close()
adj_usernames = {r[0] for r in adj_rows}

cols = ['p_participate'] + [f'P_top{n}' for n in [1, 3, 5, 8, 10]]
df_show = results[cols].copy()
df_show.index = df_show.index.map(lambda u: USERNAME_TO_PLAYER.get(u, u))
df_show = df_show.sort_values('P_top3', ascending=False).head(40)

def _highlight_adjusted(row):
    username = PLAYER_TO_USERNAME.get(row.name, '')
    if username in adj_usernames:
        return ['background-color: #fff3cd'] * len(row)
    return [''] * len(row)

display(
    df_show
    .style
    .apply(_highlight_adjusted, axis=1)
    .format({c: '{:.1%}' for c in cols})
    .set_caption(f'Top 40 by P(top3) — {TOURN_DATE}  |  Yellow = manually adjusted')
)

---
## 5 — Portfolio P&L

Update `N_TITLE_DICT` to match the event titles in your current Kalshi portfolio export.

In [ ]:
# Map event titles in the CSV → top-N thresholds
# Update the date portion (e.g. 'Jul 28') to match this week's export.
N_TITLE_DICT = {
    'Titled Tuesday Top 8: Jul 28': 8,
    'Titled Tuesday Top 5: Jul 28': 5,
    'Titled Tuesday Top 3: Jul 28': 3,
    'Titled Tuesday Winner: Jul 28': 1,
}

def _parse_avg_price(s):
    """Parse Kalshi averagePrice string (e.g. '14¢') → float (e.g. 0.14)."""
    m = re.search(r'(\d+)', str(s))
    return float(m.group()) / 100.0 if m else 0.0

try:
    kalshi_orders = pd.read_csv('../data/Kalshi-Advanced-Portfolio.csv')
    df_tt = kalshi_orders[kalshi_orders['eventTitle'].str.contains('Tuesday', na=False)].copy()
    df_tt[['position', 'volume']] = df_tt['position'].str.split(' x ', expand=True)
    df_tt['volume'] = df_tt['volume'].astype(float)
    df_tt['n'] = df_tt['eventTitle'].map(N_TITLE_DICT)
    df_tt = df_tt[df_tt['n'].notna()].copy()
    df_tt['n'] = df_tt['n'].astype(int)

    # cost = volume × order price (average paid + taker fee, matching original notebook)
    avg_p = df_tt['averagePrice'].apply(_parse_avg_price)
    df_tt['orderPrice'] = avg_p + 0.07 * avg_p * (1.0 - avg_p)
    df_tt['cost'] = df_tt['volume'] * df_tt['orderPrice']

    if df_tt.empty:
        print('No matching TT positions found — update N_TITLE_DICT above.')
    else:
        portfolio = build_portfolio(df_tt, run.players, PLAYER_TO_USERNAME)
        print(f'Portfolio: {len(df_tt)} position(s), total cost ${df_tt["cost"].sum():.2f}')
        print('Running P&L simulation (20k sims)...')
        pnl = run_portfolio_mc_score(
            run.players, run.p_play, run.hist_composites, run.hist_wts,
            portfolio, n_sims=20_000
        )
        pnl_summary(pnl)
except FileNotFoundError:
    print('Portfolio CSV not found at ../data/Kalshi-Advanced-Portfolio.csv')
    print('Export your portfolio from Kalshi and place it there.')

---
## 6 — Best trades (live Kalshi orderbook)

In [ ]:
BASE = 'https://api.elections.kalshi.com/trade-api/v2'

markets = []
for event_ticker in EVENT_TICKERS:
    cursor = None
    while True:
        params = {'event_ticker': event_ticker, 'status': 'open', 'limit': 200}
        if cursor:
            params['cursor'] = cursor
        resp = requests.get(f'{BASE}/markets', params=params)
        resp.raise_for_status()
        data = resp.json()
        markets.extend(data.get('markets', []))
        cursor = data.get('cursor')
        if not cursor:
            break

print(f'Found {len(markets)} open markets across {len(EVENT_TICKERS)} events.')

all_asks = []
for market in markets:
    ticker   = market['ticker']
    title    = market.get('title', '')
    subtitle = market.get('yes_sub_title', '')

    ob_resp = requests.get(f'{BASE}/markets/{ticker}/orderbook')
    if ob_resp.status_code != 200:
        continue
    ob = ob_resp.json().get('orderbook_fp') or {}

    for price_str, qty_str in ob.get('yes_dollars') or []:
        all_asks.append({
            'Market Ticker': ticker, 'Market Title': title, 'Player Name': subtitle,
            'Ask Price ($)': float('1.00') - float(price_str),
            'Quantity Available': float(qty_str), 'Side': 'NO',
        })
    for price_str, qty_str in ob.get('no_dollars') or []:
        all_asks.append({
            'Market Ticker': ticker, 'Market Title': title, 'Player Name': subtitle,
            'Ask Price ($)': float('1.00') - float(price_str),
            'Quantity Available': float(qty_str), 'Side': 'YES',
        })
    time.sleep(0.05)

print(f'Fetched {len(all_asks)} orderbook levels.')

In [ ]:
def _kalshi_order_price(ask):
    return ask + 0.07 * ask * (1.0 - ask)

def _parse_n(title):
    """Extract top-N from market title. 'Winner' markets have no small number — return 1."""
    m = re.search(r'\b(Top\s+)?(\d+)\b', title or '')
    if not m:
        return 1
    val = int(m.group(2))
    # If the only number in the title is large (e.g. date like 28), it's the Winner market
    return val if val <= 20 else 1

df_ob = pd.DataFrame(all_asks)
df_ob['n'] = df_ob['Market Title'].apply(_parse_n)

# Resolve player names → usernames
df_ob['username'] = df_ob['Player Name'].map(lambda n: PLAYER_TO_USERNAME.get(n))
df_ob = df_ob[df_ob['username'].notna()].copy()

evs = []
for _, row in df_ob.iterrows():
    n = int(row['n'])
    col = f'P_top{n}'
    if row['username'] not in results.index or col not in results.columns:
        evs.append(None)
        continue
    p_yes = results.loc[row['username'], col]
    evs.append(float(p_yes) if row['Side'] == 'YES' else float(1.0 - p_yes))

df_ob['EV'] = evs
df_ob = df_ob[df_ob['EV'].notna()].copy()
df_ob['Order Price'] = df_ob['Ask Price ($)'].apply(_kalshi_order_price)
df_ob['Total Cost']  = df_ob['Quantity Available'] * df_ob['Order Price']
df_ob['Expected Return'] = (df_ob['EV'] - df_ob['Order Price']) * df_ob['Quantity Available'].astype(int)
df_ob['ROI'] = (df_ob['EV'] / df_ob['Order Price']).round(3)

df_edge = df_ob[df_ob['ROI'] > 1.0].copy()
print(f'Expected return (all edge positions): ${df_edge["Expected Return"].sum():.2f}')
display(
    df_edge[['Market Title', 'Side', 'ROI', 'Order Price', 'EV', 'Quantity Available', 'Total Cost', 'Expected Return']]
    .sort_values('ROI', ascending=False)
    .style.format({
        'ROI': '{:.3f}', 'Order Price': '{:.3f}', 'EV': '{:.3f}',
        'Total Cost': '${:.2f}', 'Expected Return': '${:.2f}',
    })
)

In [ ]:
# ── Kelly sizing ──────────────────────────────────────────────────────────────
df_pos = df_edge.copy()
df_pos['kelly_f'] = (df_pos['EV'] - df_pos['Order Price']) / (1.0 - df_pos['Order Price'])

total_f = df_pos['kelly_f'].sum()
if total_f > 1.0:
    print(f'Sum of single-bet Kelly fractions = {total_f:.2f} — normalising to 1.0')
    df_pos['kelly_f'] /= total_f

df_pos['alloc_frac']   = df_pos['kelly_f'] * KELLY_MULT
df_pos['kelly_shares'] = (df_pos['alloc_frac'] * BANKROLL / df_pos['Order Price']).round().clip(lower=0).astype(int)
df_pos['kelly_cost']   = df_pos['kelly_shares'] * df_pos['Order Price']
df_pos['kelly_ev']     = df_pos['kelly_shares'] * df_pos['EV']
df_pos['kelly_edge']   = df_pos['kelly_ev'] - df_pos['kelly_cost']

df_kelly = df_pos[df_pos['kelly_shares'] > 0].copy()
total_cost = df_kelly['kelly_cost'].sum()
total_ev   = df_kelly['kelly_ev'].sum()
print(
    f'Bankroll ${BANKROLL:,.0f}  |  '
    f'Exposure ${total_cost:.0f} ({total_cost/BANKROLL*100:.1f}%)  |  '
    f'Expected edge ${total_ev - total_cost:.2f}  |  '
    f'Portfolio ROI {(total_ev/total_cost - 1)*100:.1f}%\n'
)
display(
    df_kelly[['Market Title', 'Side', 'ROI', 'Order Price', 'EV',
              'kelly_f', 'alloc_frac', 'kelly_shares', 'kelly_cost', 'kelly_edge']]
    .sort_values('kelly_f', ascending=False)
    .style.format({
        'Order Price': '{:.3f}', 'EV': '{:.3f}',
        'kelly_f': '{:.4f}', 'alloc_frac': '{:.4f}',
        'kelly_cost': '${:.2f}', 'kelly_edge': '${:.2f}',
    })
)